In [1]:
! pip install --upgrade xarray zarr gcsfs cftime nc-time-axis

  Using cached pyasn1_modules-0.4.2-py3-none-any.whl.metadata (3.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 1.4/1.4 MB 10.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   ----------------------------------- ---- 3.4/3.8 MB 16.7 MB/s eta 0:00:01
   ---------------------------------------- 3.8/3.8 MB 12.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/801.4 kB ? eta -:--:--
   --------------------------------------- 801.4/801.4 kB 11.5 MB/s eta 0:00:00
Using cached pyasn1_modules-0.4.2-py3-none-any.whl (181 kB)
   ---------------------------------------- 0.0/4.9 MB ? eta -:--:--
   ------------------------------ --------- 3.7/4.9 MB 15.6 MB/s eta 0:00:01
   ---------------------------------------- 4.9/4.9 MB 12.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/8.2 MB ? eta -

In [ ]:
# import pandas as pd
# import xarray as xr
# import gcsfs

# # 1. Load the master CMIP6 cloud catalog provided by Pangeo
# df = pd. read_csv("https://cmip6.storage.googleapis.com/pangeo-cmip6.csv")

# # 2. Filter exactly like the lecture notebook, adjusting for Ocean Acidification
# # We query for 'ph' (Sea surface pH) and 'Omon' (Ocean Monthly data table)
# # We track 'historical' baseline and 'ssp585' (the high-emissions/high-impact scenario)
# df_subset = df.query(
#     "variable_id == 'ph' & "
#     "table_id == 'Omon' & "
#     "experiment_id in ['historical', 'ssp585'] & "
#     "grid_label == 'gn'"
# )

# # Display how many matching datasets we found across different models
# print(f"Found {len(df_subset)} matching Zarr data stores.")
# print("\nAvailable Models for Ocean Acidification Analysis:")
# print(df_subset[['source_id', 'experiment_id']].drop_duplicates().head(10).to_string(index=False))

# # 3. Dynamic selection: Grab the very first available row from your filtered subset
# if len(df_subset) > 0:
#     selected_run = df_subset.iloc[0]
#     zstore_url = selected_run['zstore']
    
#     print(f"\nSuccessfully selected model: {selected_run['source_id']} ({selected_run['experiment_id']})")
#     print(f"Loading data store from: {zstore_url}")

#     # Open the cloud-hosted connection
#     fs = gcsfs.GCSFileSystem(token='anon')
#     mapper = fs.get_mapper(zstore_url)
#     ds = xr.open_zarr(mapper, consolidated=True)

#     # Look at the dataset structure
#     ds
# else:
#     print("No matching datasets found. Try widening your query filters (e.g., changing grid_label to 'gr').")

Found 191 matching Zarr data stores.

Available Models for Ocean Acidification Analysis:
      source_id experiment_id
   IPSL-CM6A-LR    historical
    CNRM-ESM2-1    historical
          CESM2    historical
    CNRM-ESM2-1        ssp585
  CanESM5-CanOE    historical
        CanESM5    historical
MPI-ESM-1-2-HAM    historical
    UKESM1-0-LL    historical
  MPI-ESM1-2-LR        ssp585
  MPI-ESM1-2-LR    historical

Successfully selected model: IPSL-CM6A-LR (historical)
Loading data store from: gs://cmip6/CMIP6/CMIP/IPSL/IPSL-CM6A-LR/historical/r8i1p1f1/Omon/ph/gn/v20180803/


In [ ]:
# # 1. Select the topmost surface layer (index 0) and the very first timestep
# # 2. Convert a small spatial slice to a DataFrame so it loads instantly without crashing
# sample_surface_slice = ds['ph'].isel(olevel=0, time=0).to_dataframe().reset_index()

# # 3. Print the resulting DataFrame view
# print("--- Surface pH DataFrame Layout ---")
# sample_surface_slice.head(10)

--- Surface pH DataFrame Layout ---


,y,x,nav_lat,nav_lon,olevel,time,ph
0,0,0,-84.210709,72.5,0.50576,1850-01-16 12:00:00,NaN
1,0,1,-84.210709,73.5,0.50576,1850-01-16 12:00:00,NaN
2,0,2,-84.210709,74.5,0.50576,1850-01-16 12:00:00,NaN
3,0,3,-84.210709,75.5,0.50576,1850-01-16 12:00:00,NaN
4,0,4,-84.210709,76.5,0.50576,1850-01-16 12:00:00,NaN
5,0,5,-84.210709,77.5,0.50576,1850-01-16 12:00:00,NaN
6,0,6,-84.210709,78.5,0.50576,1850-01-16 12:00:00,NaN
7,0,7,-84.210709,79.5,0.50576,1850-01-16 12:00:00,NaN
8,0,8,-84.210709,80.5,0.50576,1850-01-16 12:00:00,NaN
9,0,9,-84.210709,81.5,0.50576,1850-01-16 12:00:00,NaN


In [ ]:
# # 1. Slice your spatial coordinates first (this keeps it targeted)
# # Let's grab a 10x10 grid cell region in the ocean
# spatial_slice = ds['ph'].isel(olevel=0, y=slice(100, 110), x=slice(150, 160))

# # 2. CRITICAL FIX: Group by year and take the mean BEFORE converting to a dataframe
# # This reduces 1,980 monthly timesteps down to just 165 annual timesteps in memory!
# annual_spatial_slice = spatial_slice.groupby('time.year').mean(dim='time')

# # 3. Now convert the tiny, aggregated dataset to a DataFrame
# print("Crunching data... this should only take a few seconds now.")
# df_fast = annual_spatial_slice.to_dataframe().reset_index()

# # 4. Drop land rows (NaNs) so your dataset is perfectly clean
# df_fast = df_fast.dropna(subset=['ph'])

# print(f"Done! Created a clean dataframe with {len(df_fast)} rows.")
# df_fast.head(10)

In [1]:
import gcsfs
import numpy as np
import pandas as pd
import xarray as xr
import zarr
from matplotlib import pyplot as plt

xr.set_options(display_style="html")
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
plt.rcParams['figure.figsize'] = 12, 6

In [2]:
# Read the master CMIP6 cloud catalog
df = pd.read_csv(
    "https://storage.googleapis.com/cmip6/cmip6-zarr-consolidated-stores.csv"
)

# Filter for Ocean Monthly ('Omon') and Sea Surface pH ('ph') for the historical run
df_ph = df.query(
    "activity_id=='CMIP' & table_id == 'Omon' & variable_id == 'ph' & experiment_id == 'historical'"
)
df_ph.head()

,activity_id,institution_id,source_id,experiment_id,member_id,table_id,variable_id,grid_label,zstore,dcpp_init_year,version
9969,CMIP,NOAA-GFDL,GFDL-CM4,historical,r1i1p1f1,Omon,ph,gr,gs://cmip6/CMIP6/CMIP/NOAA-GFDL/GFDL-CM4/histo...,NaN,20180701
22321,CMIP,IPSL,IPSL-CM6A-LR,historical,r8i1p1f1,Omon,ph,gn,gs://cmip6/CMIP6/CMIP/IPSL/IPSL-CM6A-LR/histor...,NaN,20180803
22757,CMIP,IPSL,IPSL-CM6A-LR,historical,r9i1p1f1,Omon,ph,gn,gs://cmip6/CMIP6/CMIP/IPSL/IPSL-CM6A-LR/histor...,NaN,20180803
23794,CMIP,IPSL,IPSL-CM6A-LR,historical,r26i1p1f1,Omon,ph,gn,gs://cmip6/CMIP6/CMIP/IPSL/IPSL-CM6A-LR/histor...,NaN,20180803
24279,CMIP,IPSL,IPSL-CM6A-LR,historical,r6i1p1f1,Omon,ph,gn,gs://cmip6/CMIP6/CMIP/IPSL/IPSL-CM6A-LR/histor...,NaN,20180803
